<a href="https://colab.research.google.com/github/kseniia308/homework/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Онлайн ритейл - анализ пользователей и их заказов

## Описание проекта / Декомпозиция задачи

### Цель исследования

Определить основные категории клиентов сервиса и построить модель регрессии для цены заказа.


### Описание датасета

Датасет содержит данные о клиентах сервиса онлайн ритейла и их заказах.
Колонки:
- ***customer_id*** — идентификатор пользователя,
- ***order_date*** — дата заказа,
- ***product_id*** — идентификатор товара,
- ***category_id*** — идентифактор категории товара,
- ***category_name*** — название категории товара,
- ***product_name*** — количество баллов собственности,
- ***quantity*** — количество заказанного товара,
- ***price*** — цена за единицу товара,
- ***payment_method*** — способ оплаты,
- ***city*** — город,
- ***review_score*** — оценка товара,
- ***gender*** — пол,
- ***age*** — возраст.


###  Задачи исследования

- 1) Загрузка и описание данных:
    - подключение библиотек;
    - загрузка данных;
    - первичный осмотр данных;
    - промежуточный вывод по первому знакомству с данными;

- 2) Предобработка данных:
    - классификация пропусков (анализ их природы и соответствующая обработка);
    - поиск явных и неявных дубликатов и их обработка;
    - приведение столбцов с неправильными типами данных к соответствующему типу, если таковые имеются;
    - промежуточный вывод по предобработке;

- 3) Исследовательский анализ данных ***EDA***:
    - Анализ покупателей и заказов:
        - Распределение клиентов по демографическим признакам (пол и возраст);
        - Распределение заказов по дням недели;
        - Сравнить средний чек и частоту покупок между;
        - Предпочтения возрастных групп по дням недели;
        - Распределение заказов по месяцам;
    - Анализ выручки:
        - Топ-10 товаров по выручке и по количеству проданных единиц;
        - Распределение выручки по категориям товаров;
        - Топ-10 городов по выручке и среднему чеку;
        - Категории товаров в городах с самой большой выручкой;
        - Доли от всей выручки по городам;
        - Распределение сумм заказов по способам оплаты;
        - Анализ способов оплаты;
  - Анализ оценок:
      - Средние оценки по способам оплаты;
      - Распределение оценок;
      - Распределение цен по оценкам;
      - Средние оценки по категориям;
      - Доли низких оценок по категориям;

- 5) Задача кластеризации пользователей;


- 6) Задача регрессии;


- 7) Выводы

## Загрузка библиотек и данных, первичный осмотр данных

In [ ]:
#Подключение библиотек
import kagglehub
import os

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
#Загрузка данных из файлов
path = kagglehub.dataset_download("ertugrulesol/online-retail-data")
df = pd.read_csv(os.path.join(path, "synthetic_online_retail_data.csv"), encoding = "latin-1")

#Зададим правила вывода
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

100%|██████████| 25.5k/25.5k [00:00<00:00, 13.2MB/s]

Extracting files...


In [ ]:
#Функция для получения предварительной информации о данных
def data_info(data):
    #Получение предварительной информации о данных из датасета через метод info
    data.info()

    #Расчет доли пропущенных значений в каждом из столбцов датасета
    print()
    print('Доля пропущенных значений:')
    mis_values_visit = data.isnull().sum().to_frame('missing_values')
    mis_values_visit['Доля'] = round(data.isna().mean(),2)
    print(mis_values_visit.sort_values(by='Доля', ascending=False))
    print()

    #Получение предварительной информации о данных через отображение первых строк датасета
    display(data.head())

In [ ]:
#Получение предварительной информации о данных flights
data_info(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     1000 non-null   int64  
 1   order_date      1000 non-null   object 
 2   product_id      1000 non-null   int64  
 3   category_id     1000 non-null   int64  
 4   category_name   1000 non-null   object 
 5   product_name    1000 non-null   object 
 6   quantity        1000 non-null   int64  
 7   price           1000 non-null   float64
 8   payment_method  1000 non-null   object 
 9   city            1000 non-null   object 
 10  review_score    799 non-null    float64
 11  gender          897 non-null    object 
 12  age             1000 non-null   int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 101.7+ KB

Доля пропущенных значений:
                missing_values  Доля
review_score               201  0.20
gender                     103  0.10
product_id      

,customer_id,order_date,product_id,category_id,category_name,product_name,quantity,price,payment_method,city,review_score,gender,age
0,13542,2024-12-17,784,10,Electronics,Smartphone,2,373.36,Credit Card,New Oliviaberg,1.00,F,56
1,23188,2024-06-01,682,50,Sports & Outdoors,Soccer Ball,5,299.34,Credit Card,Port Matthew,NaN,M,59
2,55098,2025-02-04,684,50,Sports & Outdoors,Tent,5,23.00,Credit Card,West Sarah,5.00,F,64
3,65208,2024-10-28,204,40,Books & Stationery,Story Book,2,230.11,Bank Transfer,Hernandezburgh,5.00,M,34
4,63872,2024-05-10,202,20,Fashion,Skirt,4,176.72,Credit Card,Jenkinshaven,1.00,F,33


### Промежуточный вывод по исходному состоянию данных
На текущем этапе работы с данными можно заметить следующее:
    
- В данных есть пропуски в столбах 'gender'(20% значений) и 'review_score' (10% значений);


- Тип данных столбца 'order_date' не соответствует содержащимся в нём данным, требуется привести его к типу 'datetime';


- Судя по всему данные хорошего качества, но стоит проверить их на наличие дубликатов.

## Предобработка данных

### Обработка пропусков

Так как в столбце 'gender' 20% значений пропущены, то заполнение пропусков модой или медианой сильно повлияет на результаты исследования. Поэтому заполнены индикаторным значением 'Unknown':

In [ ]:
df['gender'] = df['gender'].fillna('Unknown')

Теперь заполним пропуски в столбце 'review_score'. Чтобы уменьшить возможные искажения результатов заполним пропущенные значения по категориям соответствующими медианами:

In [ ]:
df['review_score'] = df['review_score'].fillna(
    df.groupby('category_name')['review_score'].transform('median')
)

### Поиск дубликатов

Для начала проверим данные на наличие явных дубликатов:

In [ ]:
#Функция для подсчёта дубликатов
def data_duplicates_sum(data, name):
    if data.duplicated().sum() > 0:
        print(f'Количество дубликатов в таблице {name}: {data.duplicated().sum()}')
    else:
        print(f'В таблице {name} явных дубликатов нет')

In [ ]:
#Проверка на наличие явных дубликатов в датафрейме
data_duplicates_sum(data=df, name='df')

В таблице df явных дубликатов нет


Теперь проверим столбцы 'category_name', 'product_name', 'gender' и 'payment_method' на наличие неявных дубликатов:

In [ ]:
#Проверка на наличие неявных дубликатов в столбцах 'category_name', 'gender' и 'payment_method'
print('Уникальные значения в столбце category_name:', df['category_name'].unique())
print('Уникальные значения в столбце product_name:', df['product_name'].unique())
print('Уникальные значения в столбце gender:', df['gender'].unique())
print('Уникальные значения в столбце payment_method:', df['payment_method'].unique())

Уникальные значения в столбце category_name: ['Electronics' 'Sports & Outdoors' 'Books & Stationery' 'Fashion'
 'Home & Living']
Уникальные значения в столбце product_name: ['Smartphone' 'Soccer Ball' 'Tent' 'Story Book' 'Skirt' 'Tablet'
 'Yoga Mat' 'Pillow' 'Blanket' 'Smartwatch' 'Notebook' 'Laptop' 'Pen'
 'Shirt' 'Carpet' 'Novel' 'Eraser' 'Vase' 'Dress' 'Pants' 'T-shirt'
 'Running Shoes' 'Basketball' 'Painting' 'Headphones']
Уникальные значения в столбце gender: ['F' 'M' 'Unknown']
Уникальные значения в столбце payment_method: ['Credit Card' 'Bank Transfer' 'Cash on Delivery']


Проверим столбец 'city' на неявные дубликаты:

In [ ]:
#Проверка на наличие неявных дубликатов в столбце 'city'
print('Старое количество уникальных наименований:',len(df['city'].unique()))
print('Новое количество уникальных наименований:',len(df['city'].str.lower().str.replace(' ', '').unique()))

Старое количество уникальных наименований: 962
Новое количество уникальных наименований: 962


### Приведение столбцов к нужному типу

Приведём столбец 'order_date' к типу 'datetime':

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])

###  Промежуточный вывод после предобработки
На текущем этапе работы с данными можно заметить следующее:
- Пропуски в столбцах 'gender' и 'review_score' скорее всего возникли из-за того, что клиенты просто не указывали свой пол.

- Так как в этом столбце были пропущены 20% значений, то для того, чтобы избежать серьёзных искажений результатов, пропуски были заполнены индикаторным значением 'Unknown';

- Пропуски в столбце 'review_score' составляли 10% значений, они были заполнены медианными значениями по категориями товаров, чтобы уменьшить возможные искажения результатов;

- В данных не было явных дубликатов;

- Столбцы 'city', 'category_name', 'product_name', 'gender', и 'payment_method' были проверены на наличие неявных дубликатов, в данных их не оказалось;

- Столбец 'order_date' был приведен к соответствующему типу данных datetime.

## Исследование данных(EDA)

In [ ]:
#Сохраняем копию исходного датасета
df_original = df.copy()

### Анализ клиентов и заказов

#### Распределение клиентов по демографическим признакам

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Распределение возрастов', 'Распределение по полу'))

#Гистограмма возрастов
fig.add_trace(go.Histogram(x=df['age'], nbinsx=15, name='Age', marker_color='lightblue'), row=1, col=1)

#Столбчатая диаграмма для пола
gender_counts = df['gender'].value_counts(dropna=False).reset_index()
gender_counts.columns = ['gender', 'count']
fig.add_trace(go.Bar(x=gender_counts['gender'], y=gender_counts['count'],
                      marker_color='salmon', name='Gender'), row=1, col=2)

fig.update_layout(height=500, width=1000, title_text='Демографические признаки', showlegend=False)
fig.show()

##### Промежуточные выводы по демографическим признакам
По данным графикам можно сделать следующие выводы:

- Клиенты в целом равномерно распределены по возрасту за исключением двух возрастных групп: 15-19 и 75-79. Лидируют по количеству клиентов группы 60-64(104 человека) и 25-29(94 человека);

- Клиенты равномерно распределены по полу.

#### Распределение заказов по дням недели

In [ ]:
#Извлекаем день недели и месяц
df['day_of_week'] = df['order_date'].dt.day_name()
df['month'] = df['order_date'].dt.month_name()
df['hour'] = df['order_date'].dt.hour

In [ ]:
#Столбчатая диаграмма по дням недели
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
orders_by_day = df['day_of_week'].value_counts().reindex(day_order).reset_index()
orders_by_day.columns = ['day', 'count']

fig = px.bar(orders_by_day, x='day', y='count',
              title='Количество заказов по дням недели',
              labels={'day': 'День недели', 'count': 'Количество заказов'},
              color='count', color_continuous_scale='Sunset',
              text='count')
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.show()

##### Промежуточные выводы по распределению заказов по дням недели
По данным графикам можно сделать следующие выводы:

- В целом заказы распределены по дням недели равномерно, но видно, что в будни их больше;

- Больше всего заказов в пятницу (150), меньше всего в воскресенье (130).

#### Предпочтения возрастных групп по дням недели

In [ ]:
#Создаём столбец с возрастными группами
df['age_group'] = pd.cut(df['age'], bins=[18, 30, 45, 60, 76],
                          labels=['18-30', '31-45', '46-60', '60+'])

#Создаём сводную таблицу
day_age_crosstab = pd.crosstab(df['age_group'], df['day_of_week'], normalize='index') * 100
day_age_crosstab = day_age_crosstab[day_order]

In [ ]:
#Тепловая карта для дней недели по возрастным группам
fig = px.imshow(day_age_crosstab.round(1),
                text_auto='.1f',
                aspect='auto',
                title='Предпочтения возрастных групп по дням недели',
                labels=dict(x='День', y='Возрастная группа', color='Процент (%)'),
                color_continuous_scale='Blues')
fig.update_layout(height=400, width=700)
fig.show()

##### Промежуточные выводы по предпочтениям возрастных групп по дням недели
По данным графикам можно сделать следующие выводы:

- Для группы 18-30 самыми популярными днями были четверг (16.4%), вторник (15.5%) и воскресенье (15.5%);

- Для группы 31-45 самыми популярными днями были четверг (17.0%), понедельник (15.4%) и пятница (15.1%);

- Для группы 46-60 самыми популярными днями были среда (16.8%) и пятница (16.0%);

- Для группы 60+ самыми популярными днями были вторник (19.2%) и суббота (16.2%).

#### Распределение заказов по месяцам

In [ ]:
#Сводная таблица
months_order = ['January', 'February', 'March', 'April', 'May', 'June',
                'July', 'August', 'September', 'October', 'November', 'December']
orders_by_month = df['month'].value_counts().reindex(months_order).reset_index()
orders_by_month.columns = ['month', 'count']

In [ ]:
#График количества заказов по месяцам
months_order = ['January', 'February', 'March', 'April', 'May', 'June',
                'July', 'August', 'September', 'October', 'November', 'December']
orders_by_month = df['month'].value_counts().reindex(months_order).reset_index()
orders_by_month.columns = ['month', 'count']

fig = px.line(orders_by_month, x='month', y='count', markers=True,
              title='Количество заказов по месяцам',
              labels={'month': 'Месяц', 'count': 'Количество заказов'})
fig.update_layout(height=500, width=800)
fig.show()

##### Промежуточные выводы по распределению заказов по месяцам
По данным графикам можно сделать следующие выводы:

- Наибольшее количество заказов было сделано в декабре и январе, в мае и в период с июля по сентябрь;

- Для группы 31-45 самыми популярными днями были четверг (17.0%), понедельник (15.4%) и пятница (15.1%);

- Для группы 46-60 самыми популярными днями были среда (16.8%) и пятница (16.0%);

- Для группы 60+ самыми популярными днями были вторник (19.2%) и суббота (16.2%).

### Анализ выручки

#### Топ-10 товаров по выручке и по количеству проданных единиц

In [ ]:
#Создаём total_price с общей ценой заказа
df['total_price'] = df['quantity'] * df['price']

In [ ]:
#Топ-10 продуктов по выручке и количеству
top_revenue = df.groupby('product_name')['total_price'].sum().nlargest(10).reset_index()
top_quantity = df.groupby('product_name')['quantity'].sum().nlargest(10).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=('Топ-10 товаров по выручке', 'Топ-10 товаров по количеству'))

fig.add_trace(go.Bar(x=top_revenue['total_price'], y=top_revenue['product_name'],
                      orientation='h', marker_color='lightgreen', name='Revenue'), row=1, col=1)
fig.add_trace(go.Bar(x=top_quantity['quantity'], y=top_quantity['product_name'],
                      orientation='h', marker_color='skyblue', name='Quantity'), row=1, col=2)

fig.update_layout(height=600, width=1200, showlegend=False)
fig.show()

##### Промежуточные выводы по топ-10 товарам по выручке и количеству проданных единиц
По данным графикам можно сделать следующие выводы:

- Смартфоны лидируют и по выручке, и по количеству проданного товара, также следует отметить маты для йоги и футбольные мячи;

- Товары из категории ***"Электроника"*** заняли 5 позиций в топе по выручке (смартфоны, планшеты, ноутбуки, умные часы и наушники), из категории ***"Спорт"*** - 2 позиции (маты для йоги и футбольные мячи), из категории ***"Книги"*** - 1 позиция (тетради), из категории ***"Вещи для дома"*** - 1 позиция (вазы), из категории ***"Одежда"*** - 1 позиция (футболки);

- Товары из категории ***"Электроника"*** заняли 3 позиций в топе по количеству (смартфоны, ноутбуки и наушники), из категории ***"Спорт"*** - 3 позиции (маты для йоги, футбольные мячи и  ботинки для бега), из категории ***"Книги"*** - 2 позиции (тетради и ластики), из категории ***"Одежда"*** - 1 позиция (футболки), из категории ***"Вещи для дома"*** - 1 позиция (вазы).

#### Распределение выручки по категориям товаров

In [ ]:
#Сводная таблица по выручке категорий
cat_stats = df.groupby('category_name').agg({
    'total_price': 'sum'
}).round(2).reset_index().sort_values(by='total_price', ascending=False)


In [ ]:
#Столбчатая диаграмма для выручки по категориям
fig = px.bar(cat_stats, x='category_name', y='total_price', title='Выручка по категориям',
               color='total_price', color_continuous_scale='Reds', text='total_price',
               labels={'total_price': 'Общая выручка', 'category_name': 'Категория товаров'})
fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig.show()

##### Промежуточный вывод по распределению выручки товаров по категориям
По данному графику можно сделать следующий вывод:

- Лидером является категория "Электроника", на втором месте - "Спорт", на третьем - "Книги".

#### Топ-10 городов по выручке и среднему чеку

In [ ]:
#Сводные таблицы
city_revenue = df.groupby('city')['total_price'].sum().nlargest(10).reset_index()
city_avg_order = df.groupby('city')['total_price'].mean().nlargest(10).reset_index()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Топ-10 городов по выручке', 'Топ-10 городов по среднему чеку'))

#Топ-10 городов по выручке
fig.add_trace(go.Bar(x=city_revenue['city'], y=city_revenue['total_price'],
                      marker_color='coral', name='Revenue'), row=1, col=1)

#Топ-10 городов по среднему чеку
fig.add_trace(go.Bar(x=city_avg_order['city'], y=city_avg_order['total_price'],
                      marker_color='teal', name='Avg Order'), row=1, col=2)

fig.update_layout(height=500, width=1200, showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()

##### Промежуточный вывод по распределению выручки по городам
По данным графикам можно сделать следующий вывод:

- Видно, что города на этих двух графиках не совпадают, из чего мы можем сделать вывод о то, что в городах с самой большой выручкой средний чек меньше, но там делают больше покупок.

#### Категории товаров в городах с самой большой выручкой

In [ ]:
#Сводная таблица
city_cat_all = df.pivot_table(
    index='city',
    columns='category_name',
    values='total_price',
    aggfunc='sum',
    fill_value=0
)

In [ ]:
#Топ-10 городов по общей выручке с тепловой картой по категориям
top10_cities = df.groupby('city')['total_price'].sum().nlargest(10).index
city_cat_top10 = city_cat_all.loc[top10_cities]

fig = px.imshow(city_cat_top10,
                text_auto='.0f',
                aspect='auto',
                title='Тепловая карта по выручке: Топ-10 городов и категории',
                labels=dict(x='Категория', y='Город', color='Выручка'),
                color_continuous_scale='YlOrRd',
                height=600)
fig.show()

##### Промежуточный вывод по тепловой карте
По данному графику можно сделать следующие выводы:

- Port Melissaborough лидирует в категории "Вещи для дома", Patriciaville - в категории "Спорт", Johnsonborough - в категории "Одежда", East David - в категории "Электроника", East William - в категории "Книги";

- Больше всего выручки в этих городах было получено от категории "Вещи для дома", меньше всего - от категории "Электроника".

#### Доли от всей выручки по городам

In [ ]:
#Сводная таблица
top5_revenue = city_revenue.copy()
top5_revenue.loc[5] = ['Другие', city_revenue['total_price'].sum() - top5_revenue['total_price'].head(5).sum()]
top5_revenue = top5_revenue.head(6)

In [ ]:
#Круговая диаграмма выручки по топ-5 городам
fig = px.pie(top5_revenue, values='total_price', names='city',
              title='Распределение выручки',
              color_discrete_sequence=px.colors.sequential.RdBu)
fig.show()

##### Промежуточный вывод по долям выручки по городам
По данному графику можно сделать следующий вывод:

- На 5 городов с самой большой выручкой приходится более половина от общей выручки на платформе.

#### Распределение сумм заказов по способам оплаты

In [ ]:
#Boxplot с распределением сумм заказов по методам оплаты
fig = px.box(df, x='payment_method', y='total_price', color='payment_method',
              title='Распределение сумм заказов по способам оплаты',
              labels={'payment_method': 'Способ оплаты', 'total_price': 'Стоимость заказов'},
              color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

##### Промежуточный вывод по распределению сумм заказов по способам оплаты
По данному графику можно сделать следующие выводы:

- В целом у всех трёх способов оплаты наблюдается схожая ситуация: медианы, 1-ая и 3-ая квартили близки по своим значениям;

- Видно, что для заказов, оплаченных банковскими переводами верхняя граница ниже чем у двух других способов оплаты, и из-за этого чуть больше значений-выбросов.

#### Анализ способов оплаты

In [ ]:
#Сводная таблица
pay_counts = df['payment_method'].value_counts().reset_index()
pay_counts.columns = ['payment_method', 'count']

pay_avg_order = df.groupby('payment_method')['total_price'].mean().reset_index()

In [ ]:
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=('Распределение способов оплаты', 'Средний чек по способом оплаты'),
                     specs=[[{'type': 'domain'}, {'type': 'xy'}]])

#Распределение заказов по способам оплаты
fig.add_trace(go.Pie(labels=pay_counts['payment_method'], values=pay_counts['count'],
                      hole=0.3, name='Share'), row=1, col=1)

#Средний чек по способам оплаты
fig.add_trace(go.Bar(x=pay_avg_order['payment_method'], y=pay_avg_order['total_price'],
                      marker_color='lightcoral', name='Avg Order'), row=1, col=2)

fig.update_layout(height=500, width=1100, title_text='Анализ способов оплаты', showlegend=False)
fig.show()

##### Промежуточный вывод по анализу способов оплаты
По данному графику можно сделать следующие выводы:

- Оплата наличными встречается немного чаще, но в целом все три способа оплаты одинаково популярны;

- Средний чек у заказов, оплаченных кредитными картами, чуть выше, чем  у остальных двух способов, но они не сильно различаются.

### Анализ оценок

#### Средние оценки по способам оплаты

In [ ]:
#Сводная таблица
pay_review = df.groupby('payment_method')['review_score'].mean().reset_index()

In [ ]:
#Столбчатая диаграмма со средними оценками по способам оплаты
fig = px.bar(pay_review, x='payment_method', y='review_score',
              title='Средние оценки по способам оплаты',
              labels={'payment_method': 'Способ оплаты', 'review_score': 'Оценка'},
              color='review_score', color_continuous_scale='Viridis',
              text='review_score')
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis_range=[1, 5])
fig.show()

##### Промежуточный вывод по средним оценкам по способам оплаты
По данному графику можно сделать следующий вывод:

- Средние оценки для способов оплаты практически не отличаются.

#### Распределение оценок

In [ ]:
#Сводная таблица
review_counts = df['review_score'].value_counts(dropna=False).sort_index().reset_index()
review_counts.columns = ['review_score', 'count']


In [ ]:
#Столбчатая диаграмма с распределением оценок
fig = px.bar(review_counts, x='review_score', y='count',
              title='Распределение оценок',
              labels={'review_score': 'Оценка', 'count': 'Количество'},
              color='count', color_continuous_scale='Aggrnyl',
              text='count')
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.show()

##### Промежуточный вывод по распределению оценок
По данному графику можно сделать следующий вывод:

- Более 75% всех заказов имеют хорошую оценку 4 или 5 (781 заказ), количество заказов с плохим оценками 1 и 2 сильно меньше (110 заказов).

#### Распределение цен по оценкам

In [ ]:
#Boxplot с распределением цен по оценкам
fig = go.Figure()

for score in sorted(df['review_score'].unique()):
    subset = df[df['review_score'] == score]['price']
    fig.add_trace(go.Box(y=subset, name=f'Score {score}', boxmean='sd'))

fig.update_layout(title='Распределение цен по оценкам',
                   xaxis_title='Оценка',
                   yaxis_title='Цена единицы товара',
                   height=500,
                   width=700)
fig.show()


##### Промежуточный вывод по распределению оценок
По данному графику можно сделать следующие выводы:

- Для всех 5 оценок нет значений-выбросов;

- По графикам для оценок "1", "4", "5" видно, что у них различаются квартили, межквартильный размах, верхние и нижние границы, но при этом на каждом из трёх графиков медиана совпадает со средним;

- На графике для оценки "2" видно, что медиана меньше среднего, что говорит о наличии смещения цен в большую сторону в этой группе;

- График для оценки "3" обладает самым большим межквартильным размахом, а также видно, что медиана больше среднего, что говорит о наличии смещения цен в меньшую сторону в этой группе.

#### Средние оценки по категориям

In [ ]:
#Сводная таблица
avg_review_cat = df.groupby('category_name')['review_score'].mean().round(2).reset_index()

In [ ]:
#Столбчатая диаграмма со средними оценками по категориям
fig = px.bar(avg_review_cat, x='category_name', y='review_score',
              title='Средние оценки по категориям',
              labels={'review_score': 'Оценка', 'category_name': 'Категория'},
              color='review_score', color_continuous_scale='RdYlGn',
              range_color=[1, 5], text='review_score')
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(yaxis_range=[1, 5])
fig.show()

##### Промежуточный вывод по средним оценкам по категоряим
По данному графику можно сделать следующий вывод:

- У категории товаров "Спорт" средняя оценка заметно выше, чем у других, остальные 4 категории между собой практически не различаются.

#### Доли низких оценок по категориям

In [ ]:
#Сводная таблица
low_review_pct = df.groupby('category_name').apply(
    lambda x: (x['review_score'].between(1, 2).sum() / x['review_score'].notna().sum()) * 100
).round(1).reset_index(name='low_score_percentage').sort_values(by='low_score_percentage', ascending=False)

/tmp/ipykernel_11449/1057728142.py:2: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
#Столбчатая диаграмма с долями низких оценок (1-2) по категориям
fig = px.bar(low_review_pct, x='category_name', y='low_score_percentage',
              title='Доля низких оценок ("1" и "2") по категориям',
              labels={'low_score_percentage': 'Доля низких оценок', 'category_name': 'Категория'},
              color='low_score_percentage', color_continuous_scale='Reds',
              text='low_score_percentage')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

##### Промежуточный вывод по долям низких оценок по категоряим
По данному графику можно сделать следующий вывод:

- У категории "Вещи для дома" самая высокая доля низких оценок(13.1%), на втором месте категория "Спорт"(11.8%), остальные 3 категории между собой практически не различаются.

### Итоговый вывод по EDA

По итогам проведенного исследовательского анализа данных можно сказать следующее:


 1) Клиенты и заказы:
 - Клиенты в целом равномерно распределены по возрасту за исключением двух возрастных групп: 15-19 и 75-79. Лидируют по количеству клиентов группы 60-64(104 человека) и 25-29(94 человека);

 - Клиенты равномерно распределены по полу;

 - В целом заказы распределены по дням недели равномерно, но в будни их больше;

 - Больше всего заказов в пятницу (150), меньше всего в воскресенье (130);

 - Для возрастной группы 18-30 самыми популярными днями были четверг (16.4%), вторник (15.5%) и воскресенье (15.5%);

 - Для возрастной группы 31-45 самыми популярными днями были четверг (17.0%), понедельник (15.4%) и пятница (15.1%);

 - Для возрастной группы 46-60 самыми популярными днями были среда (16.8%) и пятница (16.0%);

 - Для возрастной группы 60+ самыми популярными днями были вторник (19.2%) и суббота (16.2%);

 - Наибольшее количество заказов было сделано в декабре и январе, в мае и в период с июля по сентябрь;

 - Для группы 31-45 самыми популярными днями были четверг (17.0%), понедельник (15.4%) и пятница (15.1%);

 - Для группы 46-60 самыми популярными днями были среда (16.8%) и пятница (16.0%);

 - Для группы 60+ самыми популярными днями были вторник (19.2%) и суббота (16.2%).

2) Выручка:
 - Смартфоны лидируют и по выручке, и по количеству проданного товара, также следует отметить маты для йоги и футбольные мячи;

 - Товары из категории ***"Электроника"*** заняли 5 позиций в топе по выручке (смартфоны, планшеты, ноутбуки, умные часы и наушники), из категории ***"Спорт"*** - 2 позиции (маты для йоги и футбольные мячи), из категории ***"Книги"*** - 1 позиция (тетради), из категории ***"Вещи для дома"*** - 1 позиция (вазы), из категории ***"Одежда"*** - 1 позиция (футболки);

 - Товары из категории ***"Электроника"*** заняли 3 позиций в топе по количеству (смартфоны, ноутбуки и наушники), из категории ***"Спорт"*** - 3 позиции (маты для йоги, футбольные мячи и  ботинки для бега), из категории ***"Книги"*** - 2 позиции (тетради и ластики), из категории ***"Одежда"*** - 1 позиция (футболки), из категории ***"Вещи для дома"*** - 1 позиция (вазы);

 - Лидером является категория "Электроника", на втором месте - "Спорт", на третьем - "Книги";

 - В городах с самой большой выручкой средний чек меньше, но там делают больше покупок;

 - Среди топ-10 городов по выручке Port Melissaborough лидирует в категории "Вещи для дома", Patriciaville - в категории "Спорт", Johnsonborough - в категории "Одежда", East David - в категории "Электроника", East William - в категории "Книги";

 - Больше всего выручки в этих городах было получено от категории "Вещи для дома", меньше всего - от категории "Электроника";

 - На 5 городов с самой большой выручкой приходится более половина от общей выручки на платформе;

 - В целом у всех трёх способов оплаты наблюдается схожая ситуация с суммами заказов: медианы, 1-ая и 3-ая квартили близки по своим значениям;

- Для заказов, оплаченных банковскими переводами верхняя граница ниже чем у двух других способов оплаты, и из-за этого чуть больше значений-выбросов;

- Оплата наличными встречается немного чаще, но в целом все три способа оплаты одинаково популярны;

- Средний чек у заказов, оплаченных кредитными картами, чуть выше, чем  у остальных двух способов, но они не сильно различаются.

3) Оценки:

- Средние оценки для способов оплаты практически не отличаются;

- Более 75% всех заказов имеют хорошую оценку 4 или 5 (781 заказ), количество заказов с плохим оценками 1 и 2 сильно меньше (110 заказов);

- Для всех 5 оценок нет значений-выбросов среди цен;

- У оценок "1", "4", "5"  различаются квартили, межквартильный размах, верхние и нижние границы, но при у всех трёх медиана совпадает со средним;

- У оценки "2"  медиана меньше среднего, что говорит о наличии смещения цен в большую сторону в этой группе;

- У оценки "3" самый большой межквартильным размахом, а также медиана больше среднего, что говорит о наличии смещения цен в меньшую сторону в этой группе;

- У категории товаров "Спорт" средняя оценка заметно выше, чем у других, остальные 4 категории между собой практически не различаются;

- У категории "Вещи для дома" самая высокая доля низких оценок(13.1%), на втором месте категория "Спорт"(11.8%), остальные 3 категории между собой практически не различаются.